# Can EmbeddingGemma 2 be trusted to categorise card transactions?

At work I build AI systems that categorise financial transactions. In an earlier study of embedding retrieval
for this, accuracy was 0.93–0.98 for merchants we'd seen before but only 0.599 for new ones, and the
confidence signals didn't flag the wrong answers. Here I test whether that holds on public data with an open model.

**The idea, as an analogy.** Imagine a cashier with a notebook of every receipt from the last three years.
When a new receipt comes in, they look up the line that *looks* most like it and copy its category.
That is all embedding retrieval does. So two questions:
* what happens when the shop on the new receipt was never in the notebook?
* when the cashier says "this looks almost identical", are they actually right?

**What I'm testing**
* **H1**: accuracy doesn't fade slowly as merchants get rarer; it falls off a cliff at *zero* prior sightings.
* **H2**: a high similarity score doesn't reliably mean the answer is right.
* **H3**: different merchants with lookalike names get high similarity ("false friends").
* **H4**: everyday descriptor noise (`SQ *` prefixes, store numbers, truncation) hurts matching.
* **H5**: a simple character n-gram matcher does about as well, because descriptors behave like names, not sentences.

**Model:** `google/embeddinggemma-2` (740M parameters, Apache 2.0)
**Data:** State of Delaware employee credit-card transactions, July 2022 – June 2026 (public domain)
**Runtime:** about 35 min on a free Colab T4 (Runtime → Change runtime type → T4 GPU → Run all)

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "huggingface_hub==1.33.0"

In [ ]:
import os, re, json, urllib.request
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score

MODEL_ID = "google/embeddinggemma-2"
MODEL_REVISION = "914f7f89142e33e77833254d9c9b90c3cef7303b"   # pin the exact model version
DATA_URL = "https://data.delaware.gov/api/views/nurt-5rqw/rows.csv?accessType=DOWNLOAD"
SPLIT_DATE = "2025-09-04"   # 80% of the transactions happen before this date (checked in step 3)
N_QUERIES = 20000
SEED = 42

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("running on", device)

## 1. Transactional Data Pre-processing
Each row is one transaction: the date, the raw merchant text as printed on the statement, and the merchant
category the card network has on file (the MCC description, e.g. "eating places restaurants").
**The category is our sole ground truth.**

In [ ]:
if not os.path.exists("delaware_cc.csv"):            # ~277 MB, only downloaded once
    urllib.request.urlretrieve(DATA_URL, "delaware_cc.csv")

df = pd.read_csv("delaware_cc.csv", usecols=["TRANS_DT", "MERCHANT", "CAT_DESCR"], dtype=str, keep_default_na=False)
df = df.rename(columns={"TRANS_DT": "date", "MERCHANT": "descriptor", "CAT_DESCR": "category"})
df["date"] = pd.to_datetime(df.date, format="%m/%d/%Y", errors="coerce")
df = df[(df.date >= "2022-07-01") & (df.date <= "2026-06-30")]       # four full fiscal years
df["descriptor"] = df.descriptor.str.strip()

# "STATIONERY-OFFICE SUPPLIES" (older years) and "Stationery Office Supplies" should be the same category
df["category"] = df.category.str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
df = df[(df.descriptor != "") & (df.category != "") & (df.category != "internal transaction")]  # no chargebacks

print(f"{len(df):,} transactions | {df.descriptor.nunique():,} different descriptor strings | {df.category.nunique()} categories")
df.sample(5, random_state=1)

## 2. Normalization of the Transactions
The data has no merchant ID, so I have derived a merchant name from the descriptor with a few readable rules:
droping the payment-processor prefix, cutting off store/order numbers, dropping off the punctuations. It isn't perfect
(`EASTERN SHORE COFFEE WA` and `EASTERN SHORE COFFEE W` stay separate), but every rule is visible and is normalized to a finite form.
The merchant name is only used to decide *"have we seen this merchant before?"*; then , the correctness is judged based on the category.

In [ ]:
PROCESSORS = "SQ|TST|PAYPAL|PP|SP|IN|PY|WPY|TOAST|GOOGLE|APLPAY|SUMUP|IZ|ZTL|CKE|FS|BT|EB|DD|PMT"
STOPWORDS = r"\b(INC|LLC|CO|CORP|LTD|COM|THE|AND)\b"

def tidy(s):
    s = re.sub(r"[^A-Z0-9 ]", "", s)                       # WAL-MART -> WALMART
    return " ".join(re.sub(STOPWORDS, " ", s).split())

def merchant_name(desc):
    s = desc.upper().strip()
    s = re.sub(rf"^({PROCESSORS})\s*\*\s*", "", s)         # SQ *SHAKE SHACK -> SHAKE SHACK
    s = re.sub(r"^(SP|EB|DD)\s+(?=\S)", "", s)             # SP VIKTOS (Shopify, no star) -> VIKTOS
    s = s.split("*")[0]                                    # AMZN MKTP US*Z58J27N20 -> AMZN MKTP US
    name = tidy(re.split(r"#|\d{3,}", s)[0])               # cut at store/order numbers: WAL-MART #2791 -> WALMART
    words = name.split()
    if len(words) > 1 and len(words[-1]) == 1 and words[-1].isalpha():
        name = " ".join(words[:-1])                        # TARGET T-1312 -> TARGET
    if sum(c.isalpha() for c in name) >= 2:
        return name
    # rare: the descriptor starts with a number, e.g. "92550 - BWI DAILY GARAGE" -> keep the words
    name = tidy(" ".join(w for w in re.sub(r"[^A-Z0-9]+", " ", s).split() if not w.isdigit()))
    if sum(c.isalpha() for c in name) >= 2:
        return name
    return " ".join(re.sub(r"[^A-Z0-9]+", " ", desc.upper()).split())

for d in ["SQ *SHAKE SHACK", "AMZN MKTP US*Z58J27N20", "WAL-MART #2791", "STAPLS0239279988000001", "THE HOME DEPOT #8440"]:
    print(f"{d:26} -> {merchant_name(d)}")

names = {d: merchant_name(d) for d in df.descriptor.unique()}
df["merchant"] = df.descriptor.map(names)
print(f"\n{df.merchant.nunique():,} merchants")

## 3. Temporal Data Split ( Training & Evaluation )
Everything before **4 Sep 2025** is the cashier's notebook (or the reference set). From the transactions after
that date I have randomly picked 20,000 "new receipts" (the queries). For each query I counted how many times its
merchant appeared in the past. That count is what H1 is about.

In [ ]:
df = df.sort_values(["date", "descriptor", "category"], kind="mergesort").reset_index(drop=True)
print("80th percentile of dates:", df.date.quantile(0.8).date())     # that's where SPLIT_DATE comes from

past = df[df.date < SPLIT_DATE]
future = df[df.date >= SPLIT_DATE]
rng = np.random.default_rng(SEED)
queries = future.iloc[np.sort(rng.choice(len(future), N_QUERIES, replace=False))].reset_index(drop=True)

def bucket(n):
    return "0" if n == 0 else "1" if n == 1 else "2-5" if n <= 5 else "6+"

queries["times_seen"] = queries.merchant.map(past.merchant.value_counts()).fillna(0).astype(int)
queries["bucket"] = queries.times_seen.map(bucket)

# the notebook: every past descriptor once, with the category it had most often
ref = (past.groupby(["descriptor", "category"]).size().rename("n").reset_index()
           .sort_values(["descriptor", "n", "category"], ascending=[True, False, True], kind="mergesort")
           .drop_duplicates("descriptor").drop(columns="n").reset_index(drop=True))
ref["merchant"] = ref.descriptor.map(names)

print(f"past: {len(past):,} transactions -> {len(ref):,} unique descriptors")
print(f"future: {len(future):,} transactions -> {len(queries):,} sampled queries")
print(f"{queries.descriptor.isin(ref.descriptor).mean():.0%} of queries are an exact repeat of a past descriptor\n")
print(queries.bucket.value_counts().reindex(["0", "1", "2-5", "6+"]).rename("queries by times the merchant was seen"))

queries.to_csv("data/queries.csv.gz", index=False)
ref.to_csv("data/reference.csv.gz", index=False)

## 4. Embed everything and look up the closest past descriptor
I have implemented the full checkpoint and its loaded in float32 (NOTE : the model card says not to use float16). As the model card
documents, queries get the `query` prompt and past descriptors get the `document` prompt. For each query
I took the single most similar past descriptor (cosine similarity) and copied its category.

In [ ]:
model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device=device, model_kwargs={"dtype": torch.float32})
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

def embed(texts, prompt):
    return model.encode(list(texts), prompt_name=prompt, batch_size=256, normalize_embeddings=True,
                        convert_to_tensor=True, show_progress_bar=len(texts) > 5000)

def nearest(q, r):
    # closest reference row for every query, in chunks so it fits in GPU memory
    idx, sim = [], []
    for i in range(0, len(q), 1024):
        s, j = (q[i:i + 1024] @ r.T).max(dim=1)
        idx.append(j.cpu())
        sim.append(s.cpu())
    return torch.cat(idx).numpy(), torch.cat(sim).numpy()

ref_emb = embed(ref.descriptor, "document")
q_emb = embed(queries.descriptor, "query")
idx, sim = nearest(q_emb, ref_emb)

queries["match"] = ref.descriptor.values[idx]
queries["match_category"] = ref.category.values[idx]
queries["similarity"] = sim
queries["correct"] = queries.match_category == queries.category
queries["same_merchant"] = ref.merchant.values[idx] == queries.merchant
queries[["descriptor", "match", "similarity", "category", "match_category", "correct"]].head()

## 5. A simple baseline: character n-grams (H5)
Same job with no neural network: by chopping off each descriptor into 3–5 character chunks and match on shared chunks (TF-IDF).
If this does about as well then the model isn't adding much for short merchant names.

In [ ]:
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, dtype=np.float32)
ref_tfidf = tfidf.fit_transform(ref.descriptor)          # fitted on the past only

def nearest_tfidf(texts):
    q = tfidf.transform(list(texts))
    idx, sim = [], []
    for i in range(0, q.shape[0], 512):
        s = (q[i:i + 512] @ ref_tfidf.T).tocsr()
        idx.append(np.asarray(s.argmax(axis=1)).ravel())
        sim.append(s.max(axis=1).toarray().ravel())
    return np.concatenate(idx), np.concatenate(sim)

lex_idx, lex_sim = nearest_tfidf(queries.descriptor)
queries["lex_correct"] = ref.category.values[lex_idx] == queries.category
queries["lex_same_merchant"] = ref.merchant.values[lex_idx] == queries.merchant

## 6. H1: Calulcating the accuracy by how often the merchant was already seen before

In [ ]:
def margin(p, n):                # 95% margin of error for a proportion
    return 1.96 * np.sqrt(p * (1 - p) / n)

order = ["0", "1", "2-5", "6+"]
table = queries.groupby("bucket").agg(n=("correct", "size"), embedding=("correct", "mean"),
                                      tfidf=("lex_correct", "mean"), same_merchant=("same_merchant", "mean")).reindex(order)
table.loc["all"] = [len(queries), queries.correct.mean(), queries.lex_correct.mean(), queries.same_merchant.mean()]
table["n"] = table.n.astype(int)
table["embedding ±"] = margin(table.embedding, table.n)
table.loc["0", "same_merchant"] = np.nan        # a never-seen merchant can't be matched to itself
table.to_csv("results/accuracy_by_bucket.csv")

seen, new = queries[queries.times_seen > 0], queries[queries.times_seen == 0]
print(f"merchant seen before: {seen.correct.mean():.1%} right category")
print(f"merchant never seen:  {new.correct.mean():.1%} right category\n")
table.round(3)

In [ ]:
x = np.arange(4)
plt.figure(figsize=(7, 4))
plt.bar(x - 0.2, table.loc[order, "embedding"], 0.4, yerr=table.loc[order, "embedding ±"], capsize=3,
        label="EmbeddingGemma 2", color="#2a78d6")
plt.bar(x + 0.2, table.loc[order, "tfidf"], 0.4, label="char n-gram TF-IDF", color="#eb6834")
plt.xticks(x, [f"{b}\n(n={n:,})" for b, n in zip(order, table.loc[order, "n"])])
plt.xlabel("times the merchant appeared before the split date")
plt.ylabel("share of queries with the right category")
plt.ylim(0, 1)
plt.title("Accuracy is flat for known merchants and drops at zero")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("results/accuracy_by_times_seen.png", dpi=150)
plt.show()

**Result (my T4 run): a cliff, not a slope.** Seen once → 92.2%, seen 2–5 times → 92.1%, seen 6+ times → 95.9%,
never seen → **50.9%**. One earlier sighting is almost as good as hundreds; zero sightings halves the accuracy.
The TF-IDF baseline shows the same cliff (91.9% → 43.1%) and is within half a point of the model on known merchants.

## 7. H2: Is having a high similarity score means that the answer is right?
there is 3 ways of proving it:
1. how similar is the top match is when the answer is right vs. wrong (AUROC 0.5 = the score that can't tell them apart, 1.0 = perfect);
2. accuracy within each similarity band for the set of txns;
3. what wld we get if only trusted matches above a cut-off threshold.

In [ ]:
for b in order + ["all"]:
    part = queries if b == "all" else queries[queries.bucket == b]
    print(f"seen {b:>3}: right answers avg {part[part.correct].similarity.mean():.3f} | "
          f"wrong answers avg {part[~part.correct].similarity.mean():.3f} | "
          f"AUROC {roc_auc_score(part.correct, part.similarity):.2f}")

bins = np.linspace(0.70, 1.0, 61)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True)
for ax, b, title in [(axes[0], "0", "Merchant never seen"), (axes[1], "6+", "Merchant seen 6+ times")]:
    part = queries[queries.bucket == b]
    ax.hist(part[part.correct].similarity, bins=bins, density=True, alpha=0.6, color="#2a78d6", label="right category")
    ax.hist(part[~part.correct].similarity, bins=bins, density=True, alpha=0.6, color="#eb6834", label="wrong category")
    ax.set_title(title)
    ax.set_xlabel("similarity of the top match")
axes[0].legend()
plt.tight_layout()
plt.savefig("results/similarity_right_vs_wrong.png", dpi=150)
plt.show()

In [ ]:
bands = pd.cut(queries.similarity, [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00], right=False)
by_band = queries.groupby(bands, observed=True).correct.agg(n="size", accuracy="mean")
by_band = by_band[by_band.n >= 20]                       # skip bands with almost no queries
print(by_band.round(3))

plt.figure(figsize=(6, 4))
mids = [b.mid for b in by_band.index]
plt.plot(mids, by_band.accuracy, "o-", color="#2a78d6")
for m, a, n in zip(mids, by_band.accuracy, by_band.n):
    plt.annotate(f"n={n:,}", (m, a), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
plt.xlabel("similarity of the top match")
plt.ylabel("share with the right category")
plt.ylim(0, 1.05)
plt.title("Past 0.90, a higher score doesn't mean a better answer")
plt.tight_layout()
plt.savefig("results/accuracy_by_similarity.png", dpi=150)
plt.show()

In [ ]:
rows = []
for t in [0.80, 0.85, 0.90, 0.95]:
    kept = queries[queries.similarity >= t]
    rows.append({"trust if similarity >=": t, "queries kept": len(kept) / len(queries),
                 "accuracy of kept": kept.correct.mean(), "wrong answers kept": int((~kept.correct).sum()),
                 "...from never-seen merchants": int((~kept.correct & (kept.bucket == "0")).sum())})
thresholds = pd.DataFrame(rows)
thresholds.to_csv("results/threshold_table.csv", index=False)
print(thresholds.round(3).to_string(index=False))

# the flip side: is a low score a good "this merchant is new" alarm?
is_new = queries.bucket == "0"
print(f"\nlow similarity as a new-merchant alarm: AUROC {roc_auc_score(is_new, -queries.similarity):.2f} "
      f"(TF-IDF score: {roc_auc_score(is_new, -lex_sim):.2f})")

**Result (my T4 run):** the score is a good *"this merchant is new"* alarm (AUROC 0.97) but a weak
*"this answer is right"* signal:
* AUROC 0.72 overall, and **0.41 for well-known merchants**, where wrong answers score as high as right ones (0.954 vs 0.955);
* accuracy stops improving above 0.90 (96.4% in the 0.90–0.95 band, 95.4% in 0.95–1.00);
* trusting only matches ≥ 0.95 drops a third of the queries and still keeps 614 wrong answers.

## 8. H3: False Similarities that steers the embedding model to make confident mistakes

In [ ]:
cols = ["bucket", "descriptor", "match", "similarity", "category", "match_category"]

# highest-similarity matches to a *different* merchant
lookalikes = queries[~queries.same_merchant].sort_values("similarity", ascending=False).drop_duplicates(["descriptor", "match"])
lookalikes[cols].to_csv("results/lookalikes.csv", index=False)
lookalikes[cols].head(15)

In [ ]:
# the most confident wrong answers in each bucket
wrong = queries[~queries.correct].sort_values("similarity", ascending=False).drop_duplicates(["descriptor", "match"])
wrong[cols].to_csv("results/wrong_answers.csv", index=False)

w6 = queries[(queries.bucket == "6+") & ~queries.correct]
print(f"wrong answers for merchants seen 6+ times: {len(w6)}, and in {w6.same_merchant.mean():.0%} of them "
      "the model found the right merchant (the category label itself changed)")
wrong.groupby("bucket").head(4).sort_values(["bucket", "similarity"], ascending=[True, False])[cols]

**Result (my T4 run):** the top "different merchant" matches are mostly the same business written differently
(`TST* ILLY S COFFEE` vs `TST* ILLY'S COFFEE`), so true false friends are rare at the top. The confident mistakes
are mostly one name carrying several categories: `SHERATON GRAND SEATTLE` (lodging) matched `SHERATON SEATTLE`
(restaurant), `DELAWARE MUSEUM OF NATUR` (attraction) matched the museum's gift shop, and `EXXON CRYSTAL HILL MAR`
is filed as both a fuel pump and a service station.

## 9. H4: does everyday descriptor noise break it?
I take 1,000 past descriptors, mess them up the way card statements do, and check whether the lookup still lands on the same merchant.

In [ ]:
rng = np.random.default_rng(SEED + 1)
pick = np.sort(rng.choice(np.flatnonzero(ref.descriptor.str.len() > 12), 1000, replace=False))
originals = ref.descriptor.values[pick]
store = rng.integers(0, 10000, 1000)
prefix = rng.integers(0, 3, 1000)
PREFIXES = ["SQ *", "TST* ", "PAYPAL *"]

noise = {
    "no change": lambda s, i: s,
    "cut to 12 chars": lambda s, i: s[:12],
    "add 'SQ *'": lambda s, i: "SQ *" + s,
    "add 'TST* '": lambda s, i: "TST* " + s,
    "add 'PAYPAL *'": lambda s, i: "PAYPAL *" + s,
    "add store number": lambda s, i: f"{s} #{store[i]:04d}",
    "prefix + store number": lambda s, i: f"{PREFIXES[prefix[i]]}{s} #{store[i]:04d}",
}
rows = []
for name, mess_up in noise.items():
    texts = [mess_up(s, i) for i, s in enumerate(originals)]
    e_idx, e_sim = nearest(embed(texts, "query"), ref_emb)
    l_idx, _ = nearest_tfidf(texts)
    rows.append({"noise": name, "example": texts[0],
                 "embedding finds merchant": (ref.merchant.values[e_idx] == ref.merchant.values[pick]).mean(),
                 "tfidf finds merchant": (ref.merchant.values[l_idx] == ref.merchant.values[pick]).mean(),
                 "embedding right category": (ref.category.values[e_idx] == ref.category.values[pick]).mean(),
                 "avg similarity": e_sim.mean()})
noise_results = pd.DataFrame(rows)
noise_results.to_csv("results/noise_results.csv", index=False)
noise_results.round(3)

**Result (my T4 run): robust.** With a prefix or a store number the merchant is still found 97.2–99.8% of the time.
Cutting to 12 characters is the worst case (91.3%, and TF-IDF gets 91.1%). What noise does do is pull the score down
(0.966 → 0.882 with prefix + store number), so a fixed trust threshold would start rejecting correct matches.

## 10. A Quick Sanity check: is it just the prompt?
The model card suggests trying with and without task prompts, so I rerun the lookup more than twice (takes ~15 min on a T4).

In [ ]:
prompt_check = {"query/document (used above)": queries.correct.values}
for name, q_prompt, d_prompt in [("no prompt", None, None), ("STS prompt on both sides", "STS", "STS")]:
    r = embed(ref.descriptor, d_prompt)
    i, _ = nearest(embed(queries.descriptor, q_prompt), r)
    prompt_check[name] = ref.category.values[i] == queries.category.values
    del r

for name, ok in prompt_check.items():
    print(f"{name:28} overall {ok.mean():.1%} | never-seen merchants {ok[is_new.values].mean():.1%}")

**Result (my T4 run):** 91.2% with the documented prompts, 91.5% with none, 91.2% with STS; on never-seen merchants
50.9%, 53.0% and 50.5%. The prompt isn't what causes the cliff.

In [ ]:
queries.to_csv("results/query_results.csv", index=False)
summary = {
    "model": MODEL_ID, "revision": MODEL_REVISION, "split_date": SPLIT_DATE, "seed": SEED, "n_queries": len(queries),
    "accuracy_by_bucket": {b: {"n": int(table.loc[b, "n"]), "embedding": float(table.loc[b, "embedding"]),
                               "tfidf": float(table.loc[b, "tfidf"])} for b in order + ["all"]},
    "new_merchant_auroc": {"embedding": float(roc_auc_score(is_new, -queries.similarity)),
                           "tfidf": float(roc_auc_score(is_new, -lex_sim))},
    "right_vs_wrong_auroc": float(roc_auc_score(queries.correct, queries.similarity)),
    "prompt_check": {name: float(ok.mean()) for name, ok in prompt_check.items()},
}
json.dump(summary, open("results/metrics.json", "w"), indent=2)
print(sorted(os.listdir("results")), sorted(os.listdir("data")))

In [ ]:
# zipping it for the FIT aplcn submi :D
!zip -qr results_and_data.zip results data
from google.colab import files
files.download("results_and_data.zip")

## What this proves
*(numbers from my Colab T4 run; the code is seeded, so a re-run gives the same results)*

| | Hypothesis | Verdict | The number that shows it |
|---|---|---|---|
| H1 | Accuracy falls off a cliff at zero prior sightings | **Confirmed** | 92.2% (seen once) → 50.9% (never seen) |
| H2 | High similarity ≠ right answer | **Confirmed** | AUROC 0.41 for well-known merchants; 614 wrong answers still above 0.95 |
| H3 | Lookalike merchants get high similarity | **Partly** | the confident mistakes are mostly one name with several categories, not two different merchants |
| H4 | Descriptor noise breaks matching | **Not really** | merchant still found 91–99.8% of the time; only the score drops |
| H5 | Character n-grams are about as good | **For known merchants** | within 0.5 points when seen before; 7.9 points worse (43.1% vs 50.9%) on new ones |

**Bottom line:** on merchant names, EmbeddingGemma 2 behaves like an excellent lookup table and a mediocre
classifier. Its similarity score is a good way to ask *"have I seen this merchant before?"* (AUROC 0.97) and a poor
way to ask *"is this answer right?"*. In practice: use the score to route new merchants somewhere else, not as a
confidence level. That matches what I saw in production (0.93–0.98 on known merchants vs 0.599 on new ones).